# 02 · Boschloo's exact test — `foe.frequentist.boschloo`

**Hexkit surface:** `pages/small_sample_analysis.py`.

| FOE function | R reference |
|---|---|
| `BoschlooEngine.exact_p_value` (wraps `scipy.stats.boschloo_exact`) | `exact2x2::boschloo` (two-sided = `tsmethod = "central"`, i.e. 2 × the smaller one-sided p, which is scipy's definition too) |
| `BoschlooEngine.newcombe_interval` (wraps statsmodels `newcomb`) | `DescTools::BinomDiffCI(method = "score")` (Newcombe hybrid score) |
| `BoschlooEngine.run` — Šidák α and decisions for A/B/n | decisions recomputed from the R p-values |

Both implementations maximise over the nuisance parameter numerically (scipy on a grid of
`sampling_points`, exact2x2 on its own grid + refinement), so p-values agree to roughly 3–4
decimals rather than to machine precision. The tolerance below reflects that.

In [1]:
import os
os.environ.setdefault("RPY2_CFFI_MODE", "ABI")  # use rpy2 ABI mode; avoids a noisy API-mode probe
%load_ext rpy2.ipython

import pandas as pd

from foe.core.models import AlternativeHypothesis as Alt, ExperimentInput
from foe.frequentist.boschloo import BoschlooEngine
from validation_helpers import Checks

checks = Checks()
engine = BoschlooEngine(max_visitors_per_arm=1_000)

## 1. Exact p-values and Newcombe intervals

In [2]:
# (name, control conv, control n, challenger conv, challenger n)
tables = [
    ("tiny email list",     3,  40,  9,  42),
    ("zero in control",     0,  25,  5,  25),
    ("both low",            2, 120,  4, 115),
    ("unequal allocation", 12,  60, 30,  90),
    ("challenger worse",   18,  50,  8,  52),
    ("identical",           7,  70,  7,  70),
]
alpha = 0.05
rows = []
for name, xc, nc, xv, nv in tables:
    for alt in ["two-sided", "greater", "less"]:
        p = engine.exact_p_value(xc, nc, xv, nv, Alt(alt))
        lo, hi = engine.newcombe_interval(xc, nc, xv, nv, alpha, Alt(alt))
        rows.append(dict(name=name, alternative=alt, xc=xc, nc=nc, xv=xv, nv=nv,
                         p_value=p, ci_low=lo, ci_high=hi))
py = pd.DataFrame(rows)
py

,name,alternative,xc,nc,xv,nv,p_value,ci_low,ci_high
0,tiny email list,two-sided,3,40,9,42,0.100010,-0.018006,0.292501
1,tiny email list,greater,3,40,9,42,0.050005,0.009116,inf
2,tiny email list,less,3,40,9,42,0.971691,-inf,0.267039
3,zero in control,two-sided,0,25,5,25,0.032841,0.026366,0.391310
4,zero in control,greater,0,25,5,25,0.016421,0.060878,inf
5,zero in control,less,0,25,5,25,1.000000,-inf,0.357681
6,both low,two-sided,2,120,4,115,0.572002,-0.028982,0.070772
7,both low,greater,2,120,4,115,0.286001,-0.019524,inf
8,both low,less,2,120,4,115,0.882032,-inf,0.060080
9,unequal allocation,two-sided,12,60,30,90,0.077539,-0.014200,0.264375


In [3]:
%%R -i py -i alpha -o r_out
suppressPackageStartupMessages({library(exact2x2); library(DescTools)})
r_out <- do.call(rbind, lapply(seq_len(nrow(py)), function(i) {
  s <- py[i, ]
  # exact2x2 parameterises the effect as p2 - p1: group 1 = control, group 2 = challenger,
  # so alternative = "greater" means challenger > control, as in FOE.
  p <- boschloo(s$xc, s$nc, s$xv, s$nv, alternative = sub("-", ".", s$alternative),
                tsmethod = "central")$p.value
  # One-sided intervals in FOE are the matching bound of a two-sided (1 - 2*alpha) interval.
  level <- if (s$alternative == "two-sided") 1 - alpha else 1 - 2 * alpha
  ci <- BinomDiffCI(s$xv, s$nv, s$xc, s$nc, conf.level = level, method = "score")
  lo <- max(ci[1, "lwr.ci"], -1); hi <- min(ci[1, "upr.ci"], 1)
  if (s$alternative == "greater") hi <- Inf
  if (s$alternative == "less")    lo <- -Inf
  data.frame(p_value = p, ci_low = lo, ci_high = hi)
}))
cbind(py[, c("name", "alternative")], r_out)

               name

 alternative

    p_value

       ci_low

     ci_high


0 

... (103 outputs left out)

        less

 0.54475874

         -Inf

  0.08708884

In [4]:
for i, row in py.iterrows():
    tag = f"{row['name']} | {row.alternative}"
    r = r_out.iloc[i]
    checks.add(f"Boschloo p   {tag}", row.p_value, r.p_value, atol=5e-4, rtol=5e-3)
    checks.add(f"Newcombe low {tag}", row.ci_low, r.ci_low, atol=1e-6)
    checks.add(f"Newcombe hi  {tag}", row.ci_high, r.ci_high, atol=1e-6)

## 2. `BoschlooEngine.run` on an A/B/C test (Šidák-adjusted decisions)

In [5]:
abc_n, abc_x = [150, 148, 152], [9, 21, 14]
res = engine.run(ExperimentInput(visitors=abc_n, conversions=abc_x, confidence_level=0.95))
run_py = pd.DataFrame({"p_value": [r.p_value for r in res], "alpha": [r.alpha for r in res],
                       "is_significant": [r.is_significant for r in res]})
run_py

,p_value,alpha,is_significant
0,0.023892,0.025321,True
1,0.355727,0.025321,False


In [6]:
%%R -i abc_n -i abc_x -o run_r
suppressPackageStartupMessages(library(exact2x2))
n <- unlist(abc_n); x <- unlist(abc_x)
a_sidak <- 1 - (1 - 0.05)^(1 / (length(n) - 1))
run_r <- do.call(rbind, lapply(2:length(n), function(i) {
  p <- boschloo(x[1], n[1], x[i], n[i], alternative = "two.sided", tsmethod = "central")$p.value
  data.frame(p_value = p, alpha = a_sidak, is_significant = p < a_sidak)
}))
run_r

    p_value

      alpha

 is_significant


1

 0.02389235

 0.02532057

           TRUE


2

 0.35565148

 0.02532057

          FALSE

In [7]:
for i in range(len(run_py)):
    checks.add(f"run() p-value challenger {i+1}", run_py.p_value[i], run_r.p_value.iloc[i], atol=5e-4, rtol=5e-3)
    checks.add(f"run() Sidak alpha challenger {i+1}", run_py.alpha[i], run_r.alpha.iloc[i], rtol=1e-12)
    checks.add_equal(f"run() decision challenger {i+1}", bool(run_py.is_significant[i]), bool(run_r.is_significant.iloc[i]))

## Summary

In [8]:
checks.report()

60 checks, 60 passed, 0 failed

Largest differences:
                                   check    hexkit         R    abs_diff               tolerance
              run() p-value challenger 2  0.355727  0.355651  7.5896e-05 atol=0.0005, rtol=0.005
           Boschloo p   identical | less  0.544813  0.544759 5.43694e-05 atol=0.0005, rtol=0.005
        Boschloo p   identical | greater  0.544813  0.544759 5.43694e-05 atol=0.0005, rtol=0.005
Boschloo p   zero in control | two-sided 0.0328415 0.0328325 8.92517e-06 atol=0.0005, rtol=0.005
  Boschloo p   zero in control | greater 0.0164207 0.0164163 4.46259e-06 atol=0.0005, rtol=0.005


,check,hexkit,R,abs_diff,tolerance,pass
0,Boschloo p tiny email list | two-sided,0.10001,0.10001,2.769163e-08,"atol=0.0005, rtol=0.005",True
1,Newcombe low tiny email list | two-sided,-0.018006,-0.018006,1.110223e-16,"atol=1e-06, rtol=0",True
2,Newcombe hi tiny email list | two-sided,0.292501,0.292501,1.110223e-16,"atol=1e-06, rtol=0",True
3,Boschloo p tiny email list | greater,0.050005,0.050005,1.384582e-08,"atol=0.0005, rtol=0.005",True
4,Newcombe low tiny email list | greater,0.009116,0.009116,8.326673e-17,"atol=1e-06, rtol=0",True
5,Newcombe hi tiny email list | greater,inf,inf,0.000000e+00,"atol=1e-06, rtol=0",True
6,Boschloo p tiny email list | less,0.971691,0.971688,2.802740e-06,"atol=0.0005, rtol=0.005",True
7,Newcombe low tiny email list | less,-inf,-inf,0.000000e+00,"atol=1e-06, rtol=0",True
8,Newcombe hi tiny email list | less,0.267039,0.267039,1.665335e-16,"atol=1e-06, rtol=0",True
9,Boschloo p zero in control | two-sided,0.032841,0.032833,8.925174e-06,"atol=0.0005, rtol=0.005",True
